In [1]:
import json
import os

In [2]:
json_data = {
  "company": "TechCorp",
  "employees": [
    {
      "id": 1,
      "name": "John Doe",
      "role": "Software Engineer",
      "skills": [
        "Python",
        "JavaScript",
        "React"
      ],
      "projects": [
        {
          "name": "RAG System",
          "status": "In Progress"
        },
        {
          "name": "Data Pipeline",
          "status": "Completed"
        }
      ]
    },
    {
      "id": 2,
      "name": "Jane Smith",
      "role": "Data Scientist",
      "skills": [
        "Python",
        "Machine Learning",
        "SQL"
      ],
      "projects": [
        {
          "name": "ML Model",
          "status": "In Progress"
        },
        {
          "name": "Analytics Dashboard",
          "status": "Planning"
        }
      ]
    }
  ],
  "departments": {
    "engineering": {
      "head": "Mike Johnson",
      "budget": 1000000,
      "team_size": 25
    },
    "data_science": {
      "head": "Sarah Williams",
      "budget": 750000,
      "team_size": 15
    }
  }
}

In [3]:
json_data

{'company': 'TechCorp',
 'employees': [{'id': 1,
   'name': 'John Doe',
   'role': 'Software Engineer',
   'skills': ['Python', 'JavaScript', 'React'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Jane Smith',
   'role': 'Data Scientist',
   'skills': ['Python', 'Machine Learning', 'SQL'],
   'projects': [{'name': 'ML Model', 'status': 'In Progress'},
    {'name': 'Analytics Dashboard', 'status': 'Planning'}]}],
 'departments': {'engineering': {'head': 'Mike Johnson',
   'budget': 1000000,
   'team_size': 25},
  'data_science': {'head': 'Sarah Williams',
   'budget': 750000,
   'team_size': 15}}}

In [4]:
with open('data/json_files/company_data.json','w') as f:
    json.dump(json_data, f, indent=4)

In [6]:
jsonl_data = {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123}
{"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"}
{"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99}

with open('data/json_files/events.jsonl','w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')
        


In [8]:
pip install jq

Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain_community.document_loaders import JSONLoader
import json

employee_loader = JSONLoader(
    file_path='data/json_files/company_data.json',
    jq_schema='.employees[] ',
    text_content=False
)

employee_documents = employee_loader.load()
print(f"Loaded {len(employee_documents)} employee documents.")
print(f"First employee document content: {employee_documents[0].page_content}")



Loaded 2 employee documents.
First employee document content: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}


In [13]:
import json
from typing import List
from langchain_core.documents import Document

def parse_json_file(file_path: str) -> List[Document]:
    documents = []

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for emp in data.get("employees", []):
        projects = emp.get("projects", [])

        project_texts = []
        for project in projects:
            if isinstance(project, dict):
                project_texts.append(
                    f"{project.get('name', '')}: {project.get('description', '')}"
                )
            else:
                project_texts.append(str(project))

        content = f"""
Employee Profile:
Name: {emp.get('name', '')}
Role: {emp.get('role', '')}
Skills: {', '.join(emp.get('skills', []))}
Projects: {', '.join(project_texts)}
"""

        doc = Document(
            page_content=content,
            metadata={
                "name": emp.get("name", ""),
                "role": emp.get("role", ""),
                "source": file_path,
            },
        )

        documents.append(doc)

    return documents

In [14]:
json_docs = parse_json_file("data/json_files/company_data.json")

print(f"Loaded {len(json_docs)} employees")
print(json_docs[0].page_content)

Loaded 2 employees

Employee Profile:
Name: John Doe
Role: Software Engineer
Skills: Python, JavaScript, React
Projects: RAG System: , Data Pipeline: 

